### WGAN- Designed to make training morestable and produce better quality outputs (like images).

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, LeakyReLU, Reshape, Input
from tensorflow.keras.optimizers import Adam
import tensorflow as tf


In [3]:
# Set random seed
np.random.seed(42)
tf.random.set_seed(42)

# Load and normalize MNIST dataset
(X_train, _), (_, _) = mnist.load_data()
X_train = X_train / 127.5 - 1.0 # Normalize to [-1,1]
X_train = X_train.reshape(-1, 28*28)# Set dimensions

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [4]:
# Set dimensions
img_shape=(28* 28,)
latent_dim=100

In [5]:
# Build generator

def build_generator():
  model = Sequential()
  model.add(Dense(128, input_dim=latent_dim))
  model.add(LeakyReLU(0.2))
  model.add(Dense(28*28, activation='tanh')) # output in [-1, 1]
  model.add(Reshape((784,)))
  return model

In [6]:
# Build critic

def build_critic():
  model = Sequential()
  model.add(Dense(128, input_dim=784))
  model.add(LeakyReLU(0.2))
  model.add(Dense(1)) # no sigmoid
  return model

In [7]:
# Build optimizer
optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.00005)


# Build critic
critic = build_critic()
critic.compile(loss=lambda y_true, y_pred: tf.reduce_mean(y_true * y_pred), optimizer=optimizer)

# Build Generator
generator = build_generator()

# WGAN combined model

z= Input(shape=(latent_dim,))
img = generator(z)
critic.trainable = False
validity = critic(img)
combined = Model(z, validity)
combined.compile(loss=lambda y_true, y_pred: tf.reduce_mean(y_true * y_pred), optimizer=optimizer)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
def train(epochs,batch_size = 64,sample_interval = 1000,n_critic=5,clip_value=0.01):
  valid = -np.ones((batch_size,1))  # real =-1
  fake = np.ones((batch_size,1))    #  fake = 1

  for epoch in range(epochs):
    for _ in range(n_critic):
      idx = np.random.randint(0,X_train.shape[0],batch_size)
      real_imgs = X_train[idx]

      noise = np.random.normal(0,1,(batch_size,latent_dim))
      fake_img = generator.predict(noise)

      d_loss_real = critic.train_on_batch(real_imgs,valid)
      d_loss_fake = critic.train_on_batch(fake_img,fake)
      d_loss = 0.5 * np.add(d_loss_real,d_loss_fake)

      # Clip weights
      for layer in critic.layers:
        weights = layer.get_weights()
        weights = [np.clip(w,-clip_value,clip_value) for w in weights]
        layer.set_weights(weights)

    # Train Generator
    noise = np.random.normal(0,1,(batch_size,latent_dim))
    g_loss = combined.train_on_batch(noise,valid)

    # progress output
    if epoch % 100 == 0:
      print(f"{epoch} [D loss: {d_loss:.4f} [G loss: {g_loss:.4f}]")

    if epochs % sample_interval == 0:
      sample_images(epoch)



# Image Sampling

def sample_images(epoch, n=5):
  noise = np.random.normal(0,1,(n*n,latent_dim))
  gen_imgs = generator.predict(noise)

  gen_imgs = 0.5 * gen_imgs +0.5 # Rescale to [0,1]

  fig , axs = plt.subplots(n,n)
  count = 0
  for i in range(n):
    for j in range(n):
      axs[i,j].imshow(gen_imgs[count,:,:],cmap = 'gray')
      axs[i,j].axis('off')
      count +=1
  plt.tight_layout()
  plt.savefig(f"gan_mnist_epoch_{epoch}.png")
  plt.show()
  plt.close()

In [ ]:
train(epochs = 5001, batch_size=128, sample_interval=1000)

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step  


/usr/local/lib/python3.13/dist-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Streaming output truncated to the last 5000 lines.
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 